Crear terminal e instalar dependencias

In [ ]:
python -m venv venv
source venv\Scripts\activate

pip install pandas matplotlib seaborn scikit-learn wordcloud
pip install fastapi uvicorn

Cargar y explorar los datos

El dataset FinancialPhraseBank tiene ~4.840 frases de noticias
financieras en inglés, etiquetadas por 16 expertos en finanzas.
3 clases: positive, negative, neutral

In [8]:
import pandas as pd

df = pd.read_csv("dataset/all-data.csv",
                 encoding="latin-1",         
                 header=None,                
                 names=["sentiment", "sentence"])

print(f"Total de frases: {len(df)}")
print(f"Columnas: {df.columns.tolist()}")
print(f"\nPrimeras 5 filas: \n")
print(df.head())

# Distribución de classes

print(f"\nDistribución de sentimientos: \n")
print(df["sentiment"].value_counts())
print(f"\nEn porcentajes: \n")
print(df["sentiment"].value_counts(normalize=True).apply(lambda x: f"{x:.1%}"))

# Ejemplos por clase

for sentence in ["positive", "negative", "neutral"]:
    print(f"\n{'='*50}")
    print(f"  EJEMPLOS {sentence.upper()}")
    print(f"{'='*50}")
    samples = df[df["sentiment"] == sentence]["sentence"].sample(3, random_state=42)
    for i, text in enumerate(samples, 1):
        print(f"  {i}. {text[:100]}...")

Total de frases: 4846
Columnas: ['sentiment', 'sentence']

Primeras 5 filas: 

  sentiment                                           sentence
0   neutral  According to Gran , the company has no plans t...
1   neutral  Technopolis plans to develop in stages an area...
2  negative  The international electronic industry company ...
3  positive  With the new production plant the company woul...
4  positive  According to the company 's updated strategy f...

Distribución de sentimientos: 

sentiment
neutral     2879
positive    1363
negative     604
Name: count, dtype: int64

En porcentajes: 

sentiment
neutral     59.4%
positive    28.1%
negative    12.5%
Name: proportion, dtype: object

  EJEMPLOS POSITIVE
  1. The new agreement , which expands a long-established cooperation between the companies , involves th...
  2. ( ADP News ) - Finnish handling systems provider Cargotec Oyj ( HEL : CGCBV ) announced on Friday it...
  3. The world 's biggest magazine paper maker said the program to im

Neutral domina (~59%), seguido de positive (~28%) y negative (~13%). Esto es desbalanceo de clases — el problema más común en ML aplicado.


Las frases positivas usan palabras como "growth", "profit", "increased". Las negativas usan "loss", "fell", "declined". Esto nos adelanta que un modelo basado en palabras individuales ya debería funcionar razonablemente

Análisis de palabras y longitudes

In [9]:
import re
from collections import Counter


# Stopwords: palabras tan comunes que no aportan significado
# "the", "a", "of"... aparecen en TODAS las frases
stopwords = {
    'the', 'a', 'an', 'in', 'of', 'to', 'and', 'for', 'is', 'was',
    'on', 'it', 'its', 'has', 'with', 'by', 'from', 'at', 'that',
    'this', 'are', 'as', 'be', 'or', 'will', 'have', 'had', 'were',
    'been', 'not', 'but', 'which', 'their', 'than', 'also', 'said',
    'would', 'about', 'more', 'during', 'per', 'eur', 'million',
    'company', 'year', 'period'
}

print("TOP 10 PALABRAS POR SENTIMIENTO")
print("="*50)
for sentiment in ["positive", "negative", "neutral"]:
    texts = " ".join(df[df["sentiment"] == sentiment]["sentence"].values)
    words = re.findall(r'[a-z]+', texts.lower())
    words = [w for w in words if w not in stopwords and len(w) > 2]
    top = Counter(words).most_common(10)

    print(f"\n  {sentiment.upper()}:")
    for word, count in top:
        bar = "█" * (count // 10)
        print(f"    {word:20s} {count:4d} {bar}")
        
        
# Si las frases negativas fueran mucho más largas,
# la longitud misma sería una feature útil.
df["word_count"] = df["sentence"].apply(lambda x: len(str(x).split()))

print(f"\nLONGITUD DE FRASES (palabras):")
for sentiment in ["positive", "negative", "neutral"]:
    subset = df[df["sentiment"] == sentiment]["word_count"]
    print(f"  {sentiment:10s} → media: {subset.mean():.1f} | "
          f"min: {subset.min()} | max: {subset.max()}")

TOP 10 PALABRAS POR SENTIMIENTO

  POSITIVE:
    finnish               202 ████████████████████
    net                   196 ███████████████████
    sales                 192 ███████████████████
    profit                192 ███████████████████
    mln                   128 ████████████
    operating             122 ████████████
    quarter               114 ███████████
    oyj                    97 █████████
    group                  97 █████████
    rose                   94 █████████

  NEGATIVE:
    profit                156 ███████████████
    net                   104 ██████████
    finnish               102 ██████████
    sales                  98 █████████
    operating              97 █████████
    quarter                89 ████████
    down                   83 ████████
    loss                   69 ██████
    compared               68 ██████
    decreased              68 ██████

  NEUTRAL:
    finland               228 ██████████████████████
    finnish               220 █

TF-IDF (Term Frequency x Inverse Document Frequency)

TF (Term Frequency): ¿Cuántas veces aparece esta palabra en ESTA frase?
  → "profit" aparece 2 veces → TF alto

IDF (Inverse Document Frequency): ¿Es rara esta palabra en el corpus?
  → "the" aparece en TODAS las frases → IDF bajo (no aporta info)
  → "restructuring" aparece en pocas → IDF alto (es discriminativa)

TF-IDF = TF × IDF
  → Palabras frecuentes en UNA frase pero raras en el corpus
    obtienen puntuación alta → son las más informativas.


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Limpiar texto
def clean_text(text):
    text = str(text).lower()                    # minúsculas
    text = re.sub(r'\d+\.?\d*', ' NUM ', text)  # números → token genérico
    text = re.sub(r'[^\w\s]', ' ', text)        # quitar puntuación
    text = re.sub(r'\s+', ' ', text).strip()    # espacios extra
    return text

df["clean_text"] = df["sentence"].apply(clean_text)

# Antes y después de la limpieza de texto

print("LIMPIEZA DE TEXTO - Antes vs Después:")
for i in range(3):
    print(f"  ANTES:   {df['sentence'].iloc[i][:80]}")
    print(f"  DESPUÉS: {df['clean_text'].iloc[i][:80]}")
    print()


# Vectorizador TF-IDF

tfidf = TfidfVectorizer(
    max_features=5000,   # solo las 5000 palabras más frecuentes
    ngram_range=(1, 2),  # unigrams Y bigrams ("net profit" como una feature)
    min_df=2,            # ignorar palabras que aparecen en menos de 2 docs
    max_df=0.95,         # ignorar palabras que aparecen en más del 95% de docs
    stop_words='english' # quitar stopwords automáticamente
)


# fit_transform: aprende el vocabulario Y transforma de golpe
X = tfidf.fit_transform(df["clean_text"])
y = df["sentiment"]

print(f"Matriz resultante: {X.shape}")
print(f"→ {X.shape[0]} frases × {X.shape[1]} features")

#Train/Test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y #Mantiene las proporciones de las clases en ambos sets
)

print(f"\nTrain: {X_train.shape[0]} frases")
print(f"Test:  {X_test.shape[0]} frases")
print(f"\nDistribución en train:")
print(y_train.value_counts())

LIMPIEZA DE TEXTO - Antes vs Después:
  ANTES:   According to Gran , the company has no plans to move all production to Russia , 
  DESPUÉS: according to gran the company has no plans to move all production to russia alth

  ANTES:   Technopolis plans to develop in stages an area of no less than 100,000 square me
  DESPUÉS: technopolis plans to develop in stages an area of no less than NUM NUM square me

  ANTES:   The international electronic industry company Elcoteq has laid off tens of emplo
  DESPUÉS: the international electronic industry company elcoteq has laid off tens of emplo

Matriz resultante: (4846, 5000)
→ 4846 frases × 5000 features

Train: 3876 frases
Test:  970 frases

Distribución en train:
sentiment
neutral     2303
positive    1090
negative     483
Name: count, dtype: int64


Entrenar modelos baseline

Aquí entrenamos 3 modelos clásicos y los comparamos. Empezar con baselines es una buena práctica: si después un modelo complejo no los supera, algo va mal.

Entrenamos 3 modelos clásicos:
- Logistic Regression: dibuja hiperplanos que separan clases
- Naive Bayes: asume que las palabras son independientes (ingenuo pero eficaz)
- Linear SVM: busca el hiperplano con el mayor margen de separación

class_weight='balanced' → da más peso a clases con menos datos.
Sin esto, el modelo aprende a predecir siempre "neutral" porque
es lo que minimiza el error total.

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
import numpy as np

models = {
    "Logistic Regression": LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=42
    ),
    "Naive Bayes": MultinomialNB(alpha=0.1),
    "Linear SVM": LinearSVC(
        class_weight='balanced', max_iter=5000, random_state=42
    ),
}

results = {}

for name, model in models.items():
    # Entrenar
    model.fit(X_train, y_train)

    # Evaluar en test
    y_pred = model.predict(X_test)
    acc = (y_pred == y_test).mean()

    # Cross-validation (evaluación más robusta)
    # Divide train en 5 partes, entrena con 4, evalúa con 1, rota 5 veces
    cv = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')

    results[name] = {"acc": acc, "cv_mean": cv.mean(), "cv_std": cv.std(), "preds": y_pred}

    print(f"\n{'='*50}")
    print(f"   {name}")
    print(f"   Accuracy (test):        {acc:.1%}")
    print(f"   Accuracy (CV 5-fold):   {cv.mean():.1%} (±{cv.std():.1%})")

# Mejor modelo: reporte detallado
best_name = max(results, key=lambda k: results[k]["acc"])
best_preds = results[best_name]["preds"]

print(f"\n{'='*50}")
print(f" MEJOR: {best_name} ({results[best_name]['acc']:.1%})")
print(f"{'='*50}")
print("\nClassification Report:")
print(classification_report(y_test, best_preds))

print("Matriz de confusión (filas=real, columnas=predicción):")
cm = confusion_matrix(y_test, best_preds, labels=["negative","neutral","positive"])
labels = ["negative", "neutral", "positive"]
print(f"              {'neg':>8s}  {'neu':>8s}  {'pos':>8s}")
for i, name in enumerate(labels):
    row = "  ".join(f"{v:8d}" for v in cm[i])
    print(f"  {name:10s}  {row}")




   Logistic Regression
   Accuracy (test):        72.2%
   Accuracy (CV 5-fold):   74.1% (±1.3%)

   Naive Bayes
   Accuracy (test):        72.6%
   Accuracy (CV 5-fold):   72.3% (±0.7%)

   Linear SVM
   Accuracy (test):        73.9%
   Accuracy (CV 5-fold):   75.1% (±0.5%)

 MEJOR: Linear SVM (73.9%)

Classification Report:
              precision    recall  f1-score   support

    negative       0.62      0.64      0.63       121
     neutral       0.80      0.82      0.81       576
    positive       0.66      0.61      0.63       273

    accuracy                           0.74       970
   macro avg       0.69      0.69      0.69       970
weighted avg       0.74      0.74      0.74       970

Matriz de confusión (filas=real, columnas=predicción):
                   neg       neu       pos
  negative          77        29        15
  neutral           31       474        71
  positive          17        90       166


Guía de métricas:

Accuracy: % de aciertos totales. Engañosa si hay desbalanceo.
Precision: de lo que dijo que era X, ¿cuánto era realmente X?
Recall: de lo que ERA X, ¿cuánto detectó?
F1: media armónica de precision y recall. La más fiable con desbalanceo.

Interpretabilidad

Con Logistic Regression podemos ver los coeficientes:
qué palabras empujan hacia cada sentimiento.

Esto nos permite explicar POR QUÉ el
modelo tomó una decisión.

In [12]:
# Reentrenar Logistic Regression (o reusar)
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

feature_names = tfidf.get_feature_names_out()

for i, sentiment in enumerate(lr.classes_):
    coefs = lr.coef_[i]
    # Top 10 palabras que más empujan hacia esta clase
    top_idx = np.argsort(coefs)[-10:][::-1]

    print(f"\n{'='*50}")
    print(f"  {sentiment.upper()} — Top 10 palabras más influyentes")
    print(f"{'='*50}")
    for idx in top_idx:
        score = coefs[idx]
        bar = "▓" * int(abs(score) * 8)
        print(f"    {feature_names[idx]:25s} coef={score:+.3f}  {bar}")


  NEGATIVE — Top 10 palabras más influyentes
    decreased                 coef=+3.670  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
    fell                      coef=+2.720  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
    lower                     coef=+2.418  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
    staff                     coef=+2.244  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
    fall                      coef=+2.155  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
    result                    coef=+1.939  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
    negative                  coef=+1.914  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
    fell num                  coef=+1.875  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
    decreased eur             coef=+1.853  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓
    half                      coef=+1.820  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓

  NEUTRAL — Top 10 palabras más influyentes
    value                     coef=+1.551  ▓▓▓▓▓▓▓▓▓▓▓▓
    approximately             coef=+1.391  ▓▓▓▓▓▓▓▓▓▓▓
    includes                  coef=+1.256  ▓▓▓▓▓▓▓▓▓▓
    deal                      coef=+1.127  ▓▓▓▓▓▓▓▓▓
    development               coef=+1.012  ▓▓▓▓▓▓▓▓
    sales num          

Predicción en titulares nuevos

In [13]:
import pickle

# Guardar modelo
with open("model.pkl", "wb") as f:
    pickle.dump({"model": lr, "tfidf": tfidf}, f)
print("Modelo guardado en model.pkl\n")

def predict(text):
    cleaned = clean_text(text)
    features = tfidf.transform([cleaned])
    pred = lr.predict(features)[0]
    probs = lr.predict_proba(features)[0]
    prob_dict = dict(zip(lr.classes_, probs))
    return pred, prob_dict

# Prueba de titulares
headlines = [
    "Apple reports record quarterly revenue driven by strong iPhone sales",
    "Bank announces massive layoffs affecting 10000 employees",
    "The company will present its annual report next week",
    "Tesla stock surges 15% after beating earnings expectations",
    "Operating losses widened significantly in the third quarter",
    "Revenue was flat compared to the previous quarter",
    
    # Añadir titulares si se desea
]

emojis = {"positive": "📈", "negative": "📉", "neutral": "➖"}

for h in headlines:
    pred, probs = predict(h)
    emoji = emojis[pred]
    print(f"  {emoji} {pred:8s} │ {h[:65]}")
    print(f"           │ neg:{probs.get('negative',0):.0%} "
          f"neu:{probs.get('neutral',0):.0%} "
          f"pos:{probs.get('positive',0):.0%}")
    print()

Modelo guardado en model.pkl

  📈 positive │ Apple reports record quarterly revenue driven by strong iPhone sa
           │ neg:12% neu:21% pos:68%

  📉 negative │ Bank announces massive layoffs affecting 10000 employees
           │ neg:81% neu:13% pos:6%

  ➖ neutral  │ The company will present its annual report next week
           │ neg:6% neu:81% pos:13%

  📈 positive │ Tesla stock surges 15% after beating earnings expectations
           │ neg:19% neu:27% pos:54%

  📉 negative │ Operating losses widened significantly in the third quarter
           │ neg:59% neu:21% pos:20%

  📉 negative │ Revenue was flat compared to the previous quarter
           │ neg:58% neu:24% pos:19%



API con FastAPI

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import pickle, re, time

# Cargar modelo una sola vez al arrancar
with open("model.pkl", "rb") as f:
    data = pickle.load(f)
model = data["model"]
tfidf = data["tfidf"]

# Esquemas de entrada/salida
class Request(BaseModel):
    text: str  # el titular a analizar

class Response(BaseModel):
    sentiment: str
    confidence: float
    scores: dict
    processing_time_ms: float

# Limpieza (misma función que en entrenamiento)
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\d+\.?\d*', ' NUM ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

app = FastAPI(title="Financial Sentiment API")

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=Response)
def predict(req: Request):
    start = time.time()
    cleaned = clean_text(req.text)
    features = tfidf.transform([cleaned])
    pred = model.predict(features)[0]
    probs = model.predict_proba(features)[0]
    elapsed = (time.time() - start) * 1000

    return {
        "sentiment": pred,
        "confidence": round(float(max(probs)), 4),
        "scores": dict(zip(model.classes_, [round(float(p), 4) for p in probs])),
        "processing_time_ms": round(elapsed, 2),
    }

@app.post("/predict/batch")
def predict_batch(texts: list[str]):
    return [predict(Request(text=t)) for t in texts]

# Para probar la API

Terminal 1: arrancar servidor
uvicorn api:app --reload

Terminal 2: probar con curl
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"text": "Tesla stock surges after record earnings"}'

 O abre http://localhost:8000/docs en el navegador para Swagger UI